# Guardrail Plugin

This notebook demonstrates how to build a **full-featured guardrail plugin** using the Strands Agents SDK's `HookProvider` pattern.

The plugin bundles input validation, output validation, and tool call enforcement into a single reusable component that can be attached to any agent.

## Plugin Architecture

<div style="text-align:center">
    <img src="images/plugin_architecture.png" width="85%" />
</div>

Key concepts:
- Implement `HookProvider` to create a reusable guardrail component
- Register callbacks for `BeforeInvocationEvent`, `AfterInvocationEvent`, and `BeforeToolCallEvent`
- Combine input, output, and tool call guardrails in one provider
- Configure filters, tool allowlists, and error handling via constructor

**Registration pattern:**
```python
agent = Agent(hooks=[GuardrailPlugin(input_filters=[...], output_filters=[...])])
```

## Setup

In [ ]:
# Install required packages
!pip install strands-agents strands-agents-tools --upgrade -q

In [ ]:
# Content filter classes — inline definitions (no external file needed)
from dataclasses import dataclass
from enum import Enum
from typing import Optional
import re

class Severity(Enum):
    BLOCK = 'block'
    WARN = 'warn'
    REDACT = 'redact'

@dataclass
class FilterResult:
    passed: bool
    filter_name: str
    severity: Severity
    message: Optional[str] = None
    redacted_text: Optional[str] = None

class ContentFilter:
    def __init__(self, name, severity=Severity.BLOCK):
        self.name = name
        self.severity = severity
    def evaluate(self, text):
        raise NotImplementedError

class RegexContentFilter(ContentFilter):
    def __init__(self, name, patterns, severity=Severity.BLOCK):
        super().__init__(name, severity)
        self.patterns = [re.compile(p) for p in patterns]
    def evaluate(self, text):
        for pattern in self.patterns:
            if pattern.search(text):
                if self.severity == Severity.REDACT:
                    redacted = text
                    for p in self.patterns:
                        redacted = p.sub('[REDACTED]', redacted)
                    return FilterResult(False, self.name, self.severity,
                                        f'Pattern matched: {pattern.pattern}', redacted)
                return FilterResult(False, self.name, self.severity,
                                    f'Pattern matched: {pattern.pattern}')
        return FilterResult(True, self.name, self.severity)

class KeywordContentFilter(ContentFilter):
    def __init__(self, name, keywords, severity=Severity.BLOCK):
        super().__init__(name, severity)
        self.keywords = [kw.lower() for kw in keywords]
    def evaluate(self, text):
        text_lower = text.lower()
        for keyword in self.keywords:
            if keyword in text_lower:
                return FilterResult(False, self.name, self.severity,
                                    f"Prohibited keyword: '{keyword}'")
        return FilterResult(True, self.name, self.severity)

class FormatComplianceFilter(ContentFilter):
    EXECUTION_PATTERNS = [
        re.compile(r'\b(run|execute|eval)\s*\(', re.IGNORECASE),
        re.compile(r'```\s*(bash|shell|sh)\b', re.IGNORECASE),
        re.compile(r'sudo\s+\w+', re.IGNORECASE),
    ]
    def __init__(self, name='format_compliance', severity=Severity.BLOCK):
        super().__init__(name, severity)
    def evaluate(self, text):
        for pattern in self.EXECUTION_PATTERNS:
            if pattern.search(text):
                return FilterResult(False, self.name, self.severity,
                                    'Code execution instruction detected')
        return FilterResult(True, self.name, self.severity)

def run_filters(text, filters):
    for f in filters:
        result = f.evaluate(text)
        if not result.passed:
            return result
    return None

print('Content filter classes loaded.')

In [ ]:
import logging
logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
from typing import Optional

from strands.hooks import (
    HookProvider,
    HookRegistry,
    BeforeInvocationEvent,
    AfterInvocationEvent,
    BeforeToolCallEvent,
)

# Import content filters from our shared module


## The GuardrailPlugin Class

This plugin bundles three types of validation:
1. **Input validation** — inspects user messages before model inference
2. **Output validation** — inspects model responses before returning to the user
3. **Tool call validation** — enforces a tool allowlist before tool execution

Configuration options:
- `input_filters`: List of ContentFilter instances for user input
- `output_filters`: List of ContentFilter instances for model output
- `tool_allowlist`: List of allowed tool names (None = all tools allowed)
- `fail_open`: If True, filter exceptions allow the request through

In [ ]:
class GuardrailPlugin(HookProvider):
    """A reusable HookProvider that applies content guardrails to agent input, output, and tool calls."""

    def __init__(
        self,
        input_filters: list[ContentFilter] | None = None,
        output_filters: list[ContentFilter] | None = None,
        tool_allowlist: list[str] | None = None,
        fail_open: bool = True,
    ):
        self.input_filters = input_filters or []
        self.output_filters = output_filters or []
        self.tool_allowlist = tool_allowlist
        self.fail_open = fail_open

    def register_hooks(self, registry: HookRegistry) -> None:
        """Register all guardrail callbacks with the hook registry."""
        registry.add_callback(BeforeInvocationEvent, self._validate_input)
        registry.add_callback(AfterInvocationEvent, self._validate_output)
        registry.add_callback(BeforeToolCallEvent, self._validate_tool_call)

    def _validate_input(self, event: BeforeInvocationEvent) -> None:
        """Validate user input before model inference."""
        messages = event.agent.messages
        text = self._extract_input_text(messages)
        if not text:
            return

        for content_filter in self.input_filters:
            try:
                result = content_filter.evaluate(text)
                if not result.passed:
                    self._handle_input_violation(messages, result, text)
                    return
            except Exception as e:
                if not self._handle_filter_error(e, content_filter, "input"):
                    self._block_input(messages, content_filter.name)
                    return

        self._log_decision("input", "all_filters", "passed", text)

    def _validate_output(self, event: AfterInvocationEvent) -> None:
        """Validate model output before returning to user."""
        messages = event.agent.messages
        text = self._extract_output_text(messages)
        if not text:
            return

        for content_filter in self.output_filters:
            try:
                result = content_filter.evaluate(text)
                if not result.passed:
                    self._handle_output_violation(messages, result, text)
                    return
            except Exception as e:
                if not self._handle_filter_error(e, content_filter, "output"):
                    self._block_output(messages, content_filter.name)
                    return

        self._log_decision("output", "all_filters", "passed", text)

    def _validate_tool_call(self, event: BeforeToolCallEvent) -> None:
        """Validate tool calls against the configured allowlist."""
        if self.tool_allowlist is None:
            return

        tool_name = event.tool_use.get("name", "")

        if tool_name not in self.tool_allowlist:
            reason = f"Tool '{tool_name}' is not in the allowed tools list."
            event.cancel_tool = reason
            self._log_decision("tool_call", tool_name, "blocked", tool_name)
        else:
            self._log_decision("tool_call", tool_name, "passed", tool_name)

    # --- Helper methods ---

    def _extract_input_text(self, messages: list[dict]) -> str:
        if not messages:
            return ""
        last_message = messages[-1]
        if last_message.get("role") != "user":
            return ""
        text_parts = []
        for block in last_message.get("content", []):
            if "text" in block:
                text_parts.append(block["text"])
        return " ".join(text_parts)

    def _extract_output_text(self, messages: list[dict]) -> str:
        if not messages:
            return ""
        for message in reversed(messages):
            if message.get("role") == "assistant":
                for block in message.get("content", []):
                    if "text" in block:
                        return block["text"]
        return ""

    def _handle_input_violation(self, messages, result, original_text):
        if result.severity == Severity.BLOCK:
            self._log_decision("input", result.filter_name, "blocked", original_text, result.message)
            messages.clear()
            messages.append({
                "role": "user",
                "content": [{"text": (
                    "Respond only with: I cannot process that request. "
                    "The input was blocked by a content safety filter."
                )}],
            })
        elif result.severity == Severity.REDACT:
            self._log_decision("input", result.filter_name, "redacted", original_text, result.message)
            if messages and result.redacted_text:
                last_message = messages[-1]
                if last_message.get("role") == "user":
                    last_message["content"] = [{"text": result.redacted_text}]

    def _handle_output_violation(self, messages, result, original_text):
        if result.severity == Severity.BLOCK:
            self._log_decision("output", result.filter_name, "blocked", original_text, result.message)
            self._replace_assistant_response(
                messages,
                "I'm sorry, but I cannot provide that information. "
                "The response was blocked by a content safety filter.",
            )
        elif result.severity == Severity.REDACT:
            self._log_decision("output", result.filter_name, "redacted", original_text, result.message)
            if result.redacted_text:
                self._replace_assistant_response(messages, result.redacted_text)

    def _block_input(self, messages, filter_name):
        messages.clear()
        messages.append({
            "role": "user",
            "content": [{"text": (
                "Respond only with: I cannot process that request. "
                "An internal error occurred during content validation."
            )}],
        })

    def _block_output(self, messages, filter_name):
        self._replace_assistant_response(
            messages,
            "I'm sorry, but I cannot provide a response at this time. "
            "An internal error occurred during content validation.",
        )

    def _replace_assistant_response(self, messages, new_text):
        if not messages:
            return
        for message in reversed(messages):
            if message.get("role") == "assistant":
                message["content"] = [{"text": new_text}]
                return

    def _handle_filter_error(self, error, content_filter, direction):
        if self.fail_open:
            logger.error(f"Filter error in {direction} (fail-open): {error}")
            return True
        else:
            logger.error(f"Filter error in {direction} (fail-closed): {error}")
            return False

    def _log_decision(self, direction, filter_name, action, content, message=None):
        snippet = content[:50] if content else ""
        log_msg = f"[GUARDRAIL] direction={direction} filter={filter_name} action={action} snippet='{snippet}'"
        if message:
            log_msg += f" message='{message}'"
        if action == "blocked":
            logger.warning(log_msg)
        else:
            logger.debug(log_msg)

## Demo: Testing Plugin Methods Directly

We can test the plugin's validation methods using mock events — no live model needed.

For unit testing, we call the internal `_validate_*` methods directly with mock event objects.

In [ ]:
# Mock event classes for testing
class MockAgent:
    def __init__(self, messages):
        self.messages = messages

class MockBeforeEvent:
    def __init__(self, messages):
        self.agent = MockAgent(messages)

class MockAfterEvent:
    def __init__(self, messages):
        self.agent = MockAgent(messages)

class MockToolEvent:
    def __init__(self, tool_name, tool_input=None):
        self.tool_use = {"name": tool_name, "input": tool_input or {}}
        self.cancel_tool = None


# Configure the plugin
plugin = GuardrailPlugin(
    input_filters=[
        KeywordContentFilter("prohibited_topics", ["hack", "exploit"], Severity.BLOCK),
        RegexContentFilter("input_pii", [r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"], Severity.BLOCK),
    ],
    output_filters=[
        FormatComplianceFilter("output_format", Severity.BLOCK),
        RegexContentFilter("output_pii", [r"\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b"], Severity.REDACT),
    ],
    tool_allowlist=["calculator", "web_search", "file_reader"],
    fail_open=True,
)

print(f"Plugin type: {type(plugin).__name__}")
print(f"Input filters: {[f.name for f in plugin.input_filters]}")
print(f"Output filters: {[f.name for f in plugin.output_filters]}")
print(f"Tool allowlist: {plugin.tool_allowlist}")
print(f"Fail-open: {plugin.fail_open}")

In [ ]:
# Test input BLOCK
print("Test: Input BLOCK — prohibited keyword")
messages = [{"role": "user", "content": [{"text": "How do I hack a server?"}]}]
event = MockBeforeEvent(messages)
plugin._validate_input(event)
print(f"  Result: '{event.agent.messages[0]['content'][0]['text'][:60]}...'")
assert "cannot process" in event.agent.messages[0]["content"][0]["text"]

# Test input PASS
print("\nTest: Input PASS — clean content")
messages = [{"role": "user", "content": [{"text": "What is cloud computing?"}]}]
event = MockBeforeEvent(messages)
plugin._validate_input(event)
print(f"  Result: Message unchanged (passed)")
assert event.agent.messages[0]["content"][0]["text"] == "What is cloud computing?"

# Test output REDACT
print("\nTest: Output REDACT — PII in response")
messages = [{"role": "assistant", "content": [{"text": "Contact us at support@company.com for help."}]}]
event = MockAfterEvent(messages)
plugin._validate_output(event)
result_text = event.agent.messages[0]["content"][0]["text"]
print(f"  Result: '{result_text}'")
assert "[REDACTED]" in result_text

# Test tool PASS
print("\nTest: Tool PASS — allowed tool")
event = MockToolEvent("calculator")
plugin._validate_tool_call(event)
print(f"  Result: Allowed (cancel_tool={event.cancel_tool})")
assert event.cancel_tool is None

# Test tool BLOCK
print("\nTest: Tool BLOCK — disallowed tool")
event = MockToolEvent("shell_execute")
plugin._validate_tool_call(event)
print(f"  Result: Blocked (cancel_tool='{event.cancel_tool}')")
assert event.cancel_tool is not None

## Demo: Fail-Open vs Fail-Closed Behavior

In [ ]:
class BrokenFilter(ContentFilter):
    """A filter that always raises an exception."""
    def evaluate(self, text):
        raise RuntimeError("Simulated filter failure!")


# Fail-open: broken filter doesn't crash
print("Test: Fail-open — broken filter allows request through")
fail_open_plugin = GuardrailPlugin(
    input_filters=[BrokenFilter("broken", Severity.BLOCK)],
    fail_open=True,
)
messages = [{"role": "user", "content": [{"text": "Hello world"}]}]
event = MockBeforeEvent(messages)
fail_open_plugin._validate_input(event)
print(f"  Result: Message unchanged (error caught)")
assert event.agent.messages[0]["content"][0]["text"] == "Hello world"

# Fail-closed: broken filter blocks request
print("\nTest: Fail-closed — broken filter blocks request")
fail_closed_plugin = GuardrailPlugin(
    input_filters=[BrokenFilter("broken", Severity.BLOCK)],
    fail_open=False,
)
messages = [{"role": "user", "content": [{"text": "Hello world"}]}]
event = MockBeforeEvent(messages)
fail_closed_plugin._validate_input(event)
print(f"  Result: '{event.agent.messages[0]['content'][0]['text'][:60]}...'")
assert "cannot process" in event.agent.messages[0]["content"][0]["text"]

## Attaching to a Live Agent

Register the plugin using the `hooks` parameter:

```python
agent = Agent(
    system_prompt="You are a helpful assistant.",
    hooks=[plugin],
)
```

In [ ]:
try:
    from strands import Agent
    from strands.models.bedrock import BedrockModel

    model = BedrockModel(model_id="us.anthropic.claude-sonnet-4-5-20250929-v1:0")

    agent = Agent(
        model=model,
        system_prompt="You are a helpful assistant.",
        hooks=[plugin],
    )

    print("Agent created with GuardrailPlugin attached.")
    print("Testing with a safe request...")
    response = agent("What is the capital of France?")
    print(f"  Response: {response}")

    print("\nTesting with a prohibited request...")
    response = agent("How do I hack into a system?")
    print(f"  Response: {response}")

except Exception as e:
    print(f"Skipping live agent demo: {e}")
    print("(This is expected if no model provider is configured)")

## Summary

In this notebook you learned:
1. How to implement `HookProvider` for reusable guardrail components
2. Using `register_hooks` to register typed callbacks for multiple event types
3. Combining input, output, and tool call validation in one provider
4. Configuring fail-open vs fail-closed error handling
5. Testing plugin methods with mock events

**Next Steps:** See `05_tool_call_validation.ipynb` to learn about advanced tool call guardrail patterns.